# **Fine-tuning de Mistral 7B sur MedQuAD**
**<u>Objectif:</u>** Adapter le modèle au vocabulaire médical et aux relations symptômes-pathologies.

In [ ]:
# Drive pour sauvegardes
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cloner le dépôt
!git clone https://github.com/Projet-Capstone-IA/clinical-orientation-ai.git

Cloning into 'clinical-orientation-ai'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 69 (delta 34), reused 51 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 8.80 MiB | 5.62 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [ ]:
# Se place dans le répertoire
%cd clinical-orientation-ai

[Errno 2] No such file or directory: 'clinical-orientation-ai'
/content/clinical-orientation-ai


In [ ]:
# Missing packages
!pip install bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 52.5 MB/s eta 0:00:00


## **Imports & Configurations**

In [ ]:
# Imports standards
import pandas as pd
import numpy as np
import random
import torch
from torch.utils.data import DataLoader
import warnings
import sys
import pickle
import shutil

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback


warnings.filterwarnings('ignore')

In [ ]:
# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [ ]:
# Configuration du modèle
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
MAX_LENGTH = 512
BATCH_SIZE = 4  # Ajuster selon mémoire GPU

print(f"Configuration:")
print(f"- Modèle: {MODEL_NAME}")
print(f"- Max length: {MAX_LENGTH}")
print(f"- Batch size: {BATCH_SIZE}")

Configuration:
- Modèle: mistralai/Mistral-7B-v0.1
- Max length: 512
- Batch size: 4


In [ ]:
# Vérifier le GPU disponible
print(f"GPU disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Nom du GPU: {torch.cuda.get_device_name(0)}")
    print(f"Mémoire totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Mémoire disponible: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
else:
    print("Attention: Pas de GPU détecté. L'entraînement sera très lent.")

GPU disponible: True
Nom du GPU: Tesla T4
Mémoire totale: 15.64 GB
Mémoire disponible: 0.00 GB


## **Chargement des données**

In [ ]:
# Charger les splits
train_df = pd.read_csv('data/train.csv')
val_df = pd.read_csv('data/val.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Train: {len(train_df)} exemples")
print(f"Validation: {len(val_df)} exemples")
print(f"Test: {len(test_df)} exemples")

# Aperçu
print("\nAperçu des données:")
train_df.head()

Train: 11494 exemples
Validation: 2431 exemples
Test: 2434 exemples

Aperçu des données:


,question,answer,source,focus_area,question_len,answer_len
0,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma,22,1209
1,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma,35,1607
2,Who is at risk for Glaucoma? ?,Anyone can develop glaucoma. Some people are a...,NIHSeniorHealth,Glaucoma,30,493
3,How to prevent Glaucoma ?,"At this time, we do not know how to prevent gl...",NIHSeniorHealth,Glaucoma,25,467
4,What are the symptoms of Glaucoma ?,"At first, open-angle glaucoma has no symptoms....",NIHSeniorHealth,Glaucoma,35,290


In [ ]:
# Ajouter le chemin pour importer nos modules
sys.path.append('/content/clinical-orientation-ai')

# Importer notre classe Dataset
from utils.dataset import MedQADataset

print(f"Dataset importé avec succès: \n{MedQADataset}")

Dataset importé avec succès: 
<class 'utils.dataset.MedQADataset'>


## **Chargement des datasets tokenizés**

In [ ]:
# Charger les datasets tokenizés sauvegardés
with open('data/train_dataset.pkl', 'rb') as f:
    train_dataset = pickle.load(f)

with open('data/val_dataset.pkl', 'rb') as f:
    val_dataset = pickle.load(f)

with open('data/test_dataset.pkl', 'rb') as f:
    test_dataset = pickle.load(f)

print(f"Train dataset: {len(train_dataset)} exemples")
print(f"Validation dataset: {len(val_dataset)} exemples")
print(f"Test dataset: {len(test_dataset)} exemples")

Train dataset: 11494 exemples
Validation dataset: 2431 exemples
Test dataset: 2434 exemples


In [ ]:
# Vérifier un échantillon
sample = train_dataset[0]
print(f"\nClés du dataset: {sample.keys()}")
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"labels shape: {sample['labels'].shape}")


Clés du dataset: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([512])
attention_mask shape: torch.Size([512])
labels shape: torch.Size([512])


## **Les DataLoaders**

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 2874
Validation batches: 608
Test batches: 609


In [ ]:
# Tester un batch
batch = next(iter(train_loader))
print(f"\nBatch keys: {batch.keys()}")
print(f"input_ids shape: {batch['input_ids'].shape}")


Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([4, 512])


## **Configuration du modèle avec QLoRA**

In [ ]:
# Connexion
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# Configuration 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Charger le modèle
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Modèle chargé avec quantification 4-bit")
print(f"Paramètres totals: {model.num_parameters():,}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Modèle chargé avec quantification 4-bit
Paramètres totals: 7,241,732,096


## **Configuration LoRA**

In [ ]:
# Préparer le modèle pour l'entraînement k-bit
model = prepare_model_for_kbit_training(model)

# Configuration LoRA
lora_config = LoraConfig(
    r=16,  # rang
    lora_alpha=32,  # scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # modules à adapter
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Appliquer LoRA
model = get_peft_model(model, lora_config)

print("Configuration LoRA appliquée")
print(f"Paramètres entraînables: {model.num_parameters(only_trainable=True):,}")
print(f"Total paramètres: {model.num_parameters():,}")
print(f"Pourcentage entraînable: {100 * model.num_parameters(only_trainable=True) / model.num_parameters():.2f}%")

Configuration LoRA appliquée
Paramètres entraînables: 13,631,488
Total paramètres: 7,255,363,584
Pourcentage entraînable: 0.19%


## **Configuration des arguments d'entraînement avec early stopping**

In [ ]:
# Arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,  # Augmenté
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,  # Évaluer plus souvent
    save_steps=250,  # Sauvegarder plus souvent
    learning_rate=2e-4,
    fp16=True,
    save_total_limit=3,  # Garder les 3 meilleurs checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
    report_to="none",
    logging_dir="./logs",
)

print("Arguments d'entraînement configurés:")
print(f"- Epochs: {training_args.num_train_epochs}")
print(f"- Batch size: {training_args.per_device_train_batch_size}")
print(f"- Learning rate: {training_args.learning_rate}")
print(f"- Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"- Evaluation steps: {training_args.eval_steps}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Arguments d'entraînement configurés:
- Epochs: 10
- Batch size: 4
- Learning rate: 0.0002
- Gradient accumulation: 4
- Evaluation steps: 250


In [ ]:
# Initialiser le trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("Trainer initialisé avec early stopping (patience=5)")
print("Les meilleurs poids seront sauvegardés automatiquement")

Trainer initialisé avec early stopping (patience=5)
Les meilleurs poids seront sauvegardés automatiquement


## **Lancement de l'entraînement & Sauvegardes**

In [ ]:
# Démarrer l'entraînement
trainer.train()

Step,Training Loss,Validation Loss
250,0.412450,0.416054


In [ ]:
# Sauvegarder le modèle et le tokenizer
model.save_pretrained("./mistral-medquad-final")
# tokenizer.save_pretrained("./mistral-medquad-final")

# Sauvegarder sur Google Drive
shutil.make_archive("/content/drive/MyDrive/mistral-medquad-final", 'zip', "./mistral-medquad-final")

print("Modèle sauvegardé localement et sur Drive")